In [ ]:
import pandas as pd
df = pd.read_csv(
 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)

In [ ]:
df.head()

In [ ]:
# Question 1

df['code_commune'] = (
    df['code_departement'].astype(str) +
    df['code_commune'].astype(str).str.zfill(3)
    )

df['candidat'] = ( df['nom'] +' '+ df['prenom'])

df.head()

In [ ]:
# Question 2
df['candidat'].value_counts()

candidats = df['candidat'].nunique()

print( f"En 2022, il y avait {candidats} candidats à l'élection présidentielle.")

In [ ]:
# Question 3

resultats = df.groupby("candidat")["voix"].sum().reset_index()
total_voix = result["voix"].sum()
result["score (%)"] = (result["voix"] / total_voix) * 100
result = result.sort_values(by="voix", ascending=False)

result

In [ ]:
!pip install great_tables

In [ ]:
# Question 3
from great_tables import GT

result2 = (
    df.groupby("candidat", as_index=False)["voix"]
    .sum()
    .sort_values(by="voix", ascending=False)
)

total_voix = result2["voix"].sum()
result2["score (%)"] = (result2["voix"] / total_voix) * 100

# 2. Création du tableau GT
table = (
    GT(result2)
    .tab_header(
        title="Résultats du premier tour",
    )
    .cols_label(
        candidat="Candidat",
        voix="Nombre votes (total)",
        **{"score (%)": "Score (% votes exprimés)"}
    )
    .fmt_number(
        columns="voix",
        decimals=0,
        use_seps=True
    )
    .fmt_number(
        columns="score (%)",
        decimals=2
    )
)

table


In [ ]:
# On retire les abstentions, blancs et nuls
df_votes = df[~df["candidat"].isin(["Abstentions", "Blancs", "Nuls"])].copy()

# Total des voix par département et par candidat
score_departements = (
    df_votes
    .groupby(["code_departement", "candidat"], as_index=False)["voix"]
    .sum()
    .rename(columns={"voix": "votes"})
)

# Total des votes exprimés par département
totaux_departement = (
    score_departements
    .groupby("code_departement", as_index=False)["votes"]
    .sum()
    .rename(columns={"votes": "total_votes_departement"})
)

# Ajout du total départemental et calcul du score en pourcentage
score_departements = score_departements.merge(totaux_departement, on="code_departement")
score_departements["score"] = (
    score_departements["votes"] / score_departements["total_votes_departement"] * 100
).round(2)

# Tri final
score_departements = score_departements.sort_values(
    by=["code_departement", "votes"],
    ascending=[True, False]
).reset_index(drop=True)

# Affichage du résultat
score_departements.head(20)
